In [1]:
import requests
from datetime import datetime

In [2]:
API_KEY = "aa4d98ed3d7ef33234bd02984aa2c6b7"
BASE_URL = "https://api.sportsgameodds.com/v2"
HEADERS = {"X-API-Key": API_KEY}

# ---- Input ----
sport = "Soccer"
league_name = "English Premier League"
team_1 = "Chelsea"
team_2 = "Bournemouth"
target_date = "2025-01-14"

In [3]:
# ---- Step 1: Find League ID ----
def get_league_id():
    url = f"{BASE_URL}/leagues/"
    params = {"sport": sport}
    response = requests.get(url, headers=HEADERS, params=params)
    response.raise_for_status()
    leagues = response.json().get("data", [])
    for league in leagues:
        if league_name.lower() in league["name"].lower():
            return league["id"]
    raise ValueError(f"League not found: {league_name}")

# ---- Step 2: Get Events for Date & League ----
def get_match_event(league_id):
    url = f"{BASE_URL}/events/"
    params = {
        "sport": sport,
        "league_id": league_id,
        "date": target_date
    }
    response = requests.get(url, headers=HEADERS, params=params)
    response.raise_for_status()
    events = response.json().get("data", [])

    for event in events:
        teams = [team.lower() for team in [event["home_team"], event["away_team"]]]
        if team_1.lower() in teams and team_2.lower() in teams:
            return event["id"]
    raise ValueError("Match not found for the given teams/date.")

# ---- Step 3: Get Odds ----
def get_odds(event_id):
    url = f"{BASE_URL}/odds/"
    params = {"event_id": event_id}
    response = requests.get(url, headers=HEADERS, params=params)
    response.raise_for_status()
    return response.json().get("data", [])

In [4]:
try:
    league_id = get_league_id()
    event_id = get_match_event(league_id)
    odds_data = get_odds(event_id)

    print(f"Found odds for {team_1} vs {team_2} on {target_date}:\n")
    for bookmaker in odds_data:
        print(f"Bookmaker: {bookmaker['bookmaker_name']}")
        for market in bookmaker.get("markets", []):
            print(f"  Market: {market['market_type']}")
            for outcome in market.get("outcomes", []):
                print(f"    {outcome['label']}: {outcome['odds_decimal']}")
        print("-" * 30)

except Exception as e:
    print("Error:", e)

Error: League not found: English Premier League
